In [1]:
import gymnasium as gym

import numpy as np
import time
import matplotlib
import matplotlib.pyplot as plt
import random 
import numpy as np
from collections import defaultdict
#import tensorflow as tf
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Input
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from collections import deque
 
 

env=gym.make('CartPole-v1')#,render_mode='human')

(state,_)=env.reset()

In [2]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),  # Increased size for hidden layers
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, states):
        input_list = np.array(states,dtype=np.float32)
        x=torch.from_numpy(input_list).float()
        return self.fc(x)
    

In [3]:
class DQAgent:

    def __init__(self, state_dim, action_dim, gamma=0.99, lr=0.001, batch_size=64, buffer_size=10000):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.memory = deque(maxlen=buffer_size)

        # Models
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.update_target_model()

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.loss_fn = nn.SmoothL1Loss() 
        
    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q_values = self.model(state_tensor)
            return torch.argmax(q_values).item()
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
            
    def train(self): 
        
        if len(self.memory) < self.batch_size:
            return  # Not enough samples to train

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)

        # Compute target Q-values
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            max_next_q_values = next_q_values.max(dim=1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        # Compute current Q-values
        q_values = self.model(states)
        current_q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Compute loss
        loss = self.loss_fn(current_q_values, target_q_values)

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        

In [4]:
def treatState(state,level):
    state =np.append(state,level)
    return state

In [5]:

def KL(a,b):
       
        if len(a)<len(b):
            return 1
        return np.sum(a[-100:]*np.log(a[-100:]/b))


## test

In [6]:
episodeNumber=15000
timeSteps=1000
episodesResults=[]
episodesTestResults=[]
episodesTestResultsDeviation=[]
goaldistr=np.random.normal(loc=100,scale=3,size=100)
numberOfTest=50
agent = DQAgent(5, 2,batch_size=256)


In [10]:
for episodeIndex in range(1,episodeNumber+1):
    state,_=env.reset()
    state=treatState(state,0)
    
    
   
    for timeIndex in range(timeSteps+1):
        action=agent.get_action(state)
        next_state, reward, done, _, _ = env.step(action)
        next_state=treatState(next_state,timeIndex+1)
        agent.store_transition(state, action, changeReward(reward,done,timeIndex,100), next_state, done)
        #agent.store_transition(state, action, KL(np.append(episodesResults,timeIndex),goaldistr), next_state, done)
        state=next_state
        agent.train()
        if (done or timeIndex==timeSteps-1 ):
            
            episodesResults.append(timeIndex)
            break
    
    if(episodeIndex %10 ==0):
        agent.update_target_model()
    if(episodeIndex %250 ==0):
        print("episode : ",episodeIndex)
    if(episodeIndex %500 ==0):
        tmp_epsilon=agent.epsilon
        agent.epsilon=0
        testResultList=[]
        for episodeIndex in range(numberOfTest):
            state,_=env.reset()
            state=treatState(state,0)
            for timeIndex in range(timeSteps):
                action=agent.get_action(tuple(state))
                next_state, reward, done, _, _ = env.step(action)
                next_state=treatState(next_state,timeIndex+1)
                state=next_state
                if (done):
                    testResultList.append(timeIndex)
                    break
        episodesTestResults.append(sum(testResultList)/numberOfTest)
        episodesTestResultsDeviation.append(np.std(episodesTestResults))
        agent.epsilon=tmp_epsilon

env.close()

C:\Users\mathi\AppData\Local\Temp\ipykernel_27816\3036285631.py:5: RuntimeWarning: divide by zero encountered in log
  return np.sum(a[-100:]*np.log(a[-100:]/b))
C:\Users\mathi\AppData\Local\Temp\ipykernel_27816\3036285631.py:5: RuntimeWarning: invalid value encountered in multiply
  return np.sum(a[-100:]*np.log(a[-100:]/b))


[11 12 36 ...  7  8  0]
[11 12 36 ...  7  8  1]
[11 12 36 ...  7  8  2]
[11 12 36 ...  7  8  3]
[11 12 36 ...  7  8  4]
[11 12 36 ...  7  8  5]
[11 12 36 ...  7  8  6]
[11 12 36 ...  7  8  7]
[11 12 36 ...  7  8  8]
[11 12 36 ...  8  8  0]
[11 12 36 ...  8  8  1]
[11 12 36 ...  8  8  2]
[11 12 36 ...  8  8  3]
[11 12 36 ...  8  8  4]
[11 12 36 ...  8  8  5]
[11 12 36 ...  8  8  6]
[11 12 36 ...  8  8  7]
[11 12 36 ...  8  8  8]
[11 12 36 ...  8  8  9]
[11 12 36 ...  8  9  0]
[11 12 36 ...  8  9  1]
[11 12 36 ...  8  9  2]
[11 12 36 ...  8  9  3]
[11 12 36 ...  8  9  4]
[11 12 36 ...  8  9  5]
[11 12 36 ...  8  9  6]
[11 12 36 ...  8  9  7]
[11 12 36 ...  8  9  8]
[11 12 36 ...  8  9  9]
[11 12 36 ...  8  9 10]
[11 12 36 ...  9 10  0]
[11 12 36 ...  9 10  1]
[11 12 36 ...  9 10  2]
[11 12 36 ...  9 10  3]
[11 12 36 ...  9 10  4]
[11 12 36 ...  9 10  5]
[11 12 36 ...  9 10  6]
[11 12 36 ...  9 10  7]
[11 12 36 ... 10  7  0]
[11 12 36 ... 10  7  1]
[11 12 36 ... 10  7  2]
[11 12 36 ... 10

KeyboardInterrupt: 

In [8]:
goal6LearnerEp=episodesResults
%store goal6LearnerEp

DQN2episodesTestResults=episodesTestResults
%store DQN2episodesTestResults

DQN2episodesTestResultsDeviation=episodesTestResultsDeviation
%store DQN2episodesTestResultsDeviation

Stored 'goal6LearnerEp' (list)
Stored 'DQN2episodesTestResults' (list)
Stored 'DQN2episodesTestResultsDeviation' (list)


In [9]:
print(episodesTestResultsDeviation)
print(episodesTestResults)
print(len(episodesResults))

[0.0, 0.0600000000000005, 0.049888765156986245, 0.04358898943540703, 0.05878775382679628, 0.06368324391514271, 0.06916410535377254, 0.06887488656977957, 0.06531972647421809, 0.07166589146867582, 0.06994685113335043, 0.08556998435328979, 0.08515909232675747, 0.10017331919004711, 0.10184083442095118, 0.10582267006648458, 0.10292604930870332, 0.1031180553217204, 0.10784209241992194, 0.10561723344227521, 0.10307736565637313, 0.10097458971738926, 0.10143768049681895, 0.0994673313203891, 0.10450339707397102, 0.11051520016475416, 0.1103131842902649, 0.1104813217199163, 0.11276231144439637, 0.11108355213781923]
[8.46, 8.34, 8.38, 8.38, 8.5, 8.32, 8.5, 8.34, 8.38, 8.28, 8.44, 8.2, 8.46, 8.16, 8.24, 8.2, 8.38, 8.46, 8.18, 8.3, 8.34, 8.38, 8.46, 8.38, 8.16, 8.56, 8.46, 8.24, 8.52, 8.32]
15000
